# Práctica 3

In [1]:
# Librerías Básicas
import numpy as np
import pandas as pd
import math

#Procesamiento de Texto y Configuración
from textblob import TextBlob
from sklearn import set_config
set_config(display='diagram') 

# Visualización de Datos
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import cufflinks as cf
cf.set_config_file(theme='solar', offline=True)

# Preprocesamiento y Pipelines
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler

# Modelos de Regresión (Continuo)
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet, 
    Lars, BayesianRidge, SGDRegressor
)

# Modelos de Clasificación (Discreto)
from sklearn.linear_model import LogisticRegression

#Métricas de Evaluación
# Para Regresión
from sklearn.metrics import (
    r2_score, mean_absolute_error, 
    mean_absolute_percentage_error, mean_squared_error,
    accuracy_score, precision_score, recall_score, 
    f1_score, roc_auc_score, confusion_matrix, 
    classification_report, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

# Para Clasificación
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
# Enviroment setup
set_config(display="diagram")
pd.set_option("display.max_columns", 50)
pd.set_option('display.float_format', lambda x: '%.8f' % x)

In [2]:
df_train = pd.read_csv("train_p3.csv")
df_train

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,18125,220000,2,3,2,32,0,0,0,0,0,0,209259,192365,184198,171391,178742,164793,7000,15000,6000,10000,10000,7000,0
1,3983,220000,2,2,1,41,2,0,0,2,0,0,27094,27819,30363,29579,48933,97187,1465,3009,0,20000,50000,1240,0
2,19251,80000,2,1,2,27,-1,-1,-1,0,-1,-1,3199,1205,917,917,702,3099,1325,917,0,702,3099,0,0
3,4024,20000,2,2,2,38,1,2,2,4,3,2,10683,12729,14734,14190,13721,13848,2500,2501,0,0,441,1,0
4,20610,100000,2,1,2,28,0,0,0,0,0,0,102697,100462,101879,100999,101188,99328,4600,4000,4000,4000,4000,4400,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20995,13019,110000,1,1,2,28,0,0,0,0,0,0,102282,104196,105922,108820,66883,68083,5200,5000,5000,2100,2000,2000,0
20996,27065,420000,2,2,2,25,-1,0,0,0,0,0,28206,122212,117835,112705,111357,108306,100000,4013,3822,4086,3800,3600,0
20997,5873,50000,1,2,1,34,0,0,0,0,0,0,48976,47404,46338,14234,19545,20173,2023,3105,3119,11000,909,3000,0
20998,21449,100000,2,1,2,27,-2,-2,-2,-2,-2,-2,-2000,5555,0,0,0,0,7555,0,0,0,0,0,0


### Diccionario de datos:

ID: ID of each client

LIMIT_BAL: Amount of given credit in NT dollars (includes individual and family/supplementary credit

SEX: Gender (1=male, 2=female)

EDUCATION: (1=graduate school, 2=university, 3=high school, 4=others, 5=unknown, 6=unknown)

MARRIAGE: Marital status (1=married, 2=single, 3=others)

AGE: Age in years

PAY_0: Repayment status in September, 2005 (-1=pay duly, 1=payment delay for one month, 2=payment delay for two months, … 8=payment delay for eight months, 9=payment delay for nine months and above)

PAY_2: Repayment status in August, 2005 (scale same as above)

PAY_3: Repayment status in July, 2005 (scale same as above)

PAY_4: Repayment status in June, 2005 (scale same as above)

PAY_5: Repayment status in May, 2005 (scale same as above)

PAY_6: Repayment status in April, 2005 (scale same as above)

BILL_AMT1: Amount of bill statement in September, 2005 (NT dollar)

BILL_AMT2: Amount of bill statement in August, 2005 (NT dollar)

BILL_AMT3: Amount of bill statement in July, 2005 (NT dollar)

BILL_AMT4: Amount of bill statement in June, 2005 (NT dollar)

BILL_AMT5: Amount of bill statement in May, 2005 (NT dollar)

BILL_AMT6: Amount of bill statement in April, 2005 (NT dollar)

PAY_AMT1: Amount of previous payment in September, 2005 (NT dollar)

PAY_AMT2: Amount of previous payment in August, 2005 (NT dollar)

PAY_AMT3: Amount of previous payment in July, 2005 (NT dollar)

PAY_AMT4: Amount of previous payment in June, 2005 (NT dollar)

PAY_AMT5: Amount of previous payment in May, 2005 (NT dollar)

PAY_AMT6: Amount of previous payment in April, 2005 (NT dollar)

default.payment.next.month: Default payment (1=yes, 0=no)


In [ ]:
ls_cont = ["LIMIT_BAL","AGE","BILL_AMT1","BILL_AMT2","BILL_AMT3","BILL_AMT4","BILL_AMT5","BILL_AMT6",
           "PAY_AMT1","PAY_AMT2","PAY_AMT3","PAY_AMT4","PAY_AMT5","PAY_AMT6"]
ls_disc = ["SEX","MARRIAGE","EDUCATION"]
ls_indx = ["ID"]
ls_ord=["PAY_0","PAY_2","PAY_3","PAY_4","PAY_5","PAY_6"]
target = ["default payment next month"]
ls_cont,ls_disc,ls_indx,ls_ord,target

(['LIMIT_BAL',
  'AGE',
  'BILL_AMT1',
  'BILL_AMT2',
  'BILL_AMT3',
  'BILL_AMT4',
  'BILL_AMT5',
  'BILL_AMT6',
  'PAY_AMT1',
  'PAY_AMT2',
  'PAY_AMT3',
  'PAY_AMT4',
  'PAY_AMT5',
  'PAY_AMT6'],
 ['SEX', 'MARRIAGE', 'EDUCATION'],
 ['ID'],
 ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6'],
 ['default payment next month'])

In [4]:
df_train = df_train.set_index(ls_indx)
df_train.sample(5)

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
ID,,,,,,,,,,,,,,,,,,,,,,,,
11032,100000,2,3,1,43,2,2,2,2,2,2,50485,51582,52141,52691,54199,53176,2200,2000,2000,2500,0,4500,1
14630,200000,2,1,2,30,-1,-1,-1,-1,-1,-1,105,271,588,111,305,432,271,588,111,305,432,0,1
7878,260000,2,1,1,31,0,0,0,0,0,0,50312,128072,55359,39189,49909,20782,85387,33061,5039,20154,2854,29,0
25370,50000,2,2,1,41,0,0,0,0,0,0,22824,22664,5631,5933,6433,6828,1100,1100,1100,600,500,300,0
281,390000,1,3,2,35,0,0,0,0,0,0,55213,59122,72930,76543,78143,77763,10000,20000,10000,8000,6000,5000,0


In [5]:
df_train.describe()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
count,21000.00000000,21000.00000000,21000.00000000,21000.00000000,21000.00000000,21000.00000000,21000.00000000,21000.00000000,21000.00000000,21000.00000000,21000.00000000,21000.00000000,21000.00000000,21000.00000000,21000.00000000,21000.00000000,21000.00000000,21000.00000000,21000.00000000,21000.00000000,21000.00000000,21000.00000000,21000.00000000,21000.00000000
mean,167866.28571429,1.60519048,1.85228571,1.55233333,35.50209524,-0.01709524,-0.13471429,-0.16866667,-0.22228571,-0.26671429,-0.28961905,51301.89700000,49211.55866667,47090.61552381,43443.81257143,40481.23171429,39041.15371429,5651.38004762,5914.73866667,5306.84995238,4760.82866667,4788.35633333,5239.21276190,0.21690476
std,130184.88183637,0.48882138,0.78982862,0.52166208,9.21103570,1.12508473,1.20189013,1.19912707,1.17424239,1.13587440,1.15395528,73913.94893193,71270.19376317,69729.73408859,64601.44736244,61065.85731619,60219.02468601,16167.07574078,22482.43885578,18342.90525402,15170.45447926,15315.29734049,17978.87701706,0.41214703
min,10000.00000000,1.00000000,0.00000000,0.00000000,21.00000000,-2.00000000,-2.00000000,-2.00000000,-2.00000000,-2.00000000,-2.00000000,-15308.00000000,-69777.00000000,-157264.00000000,-170000.00000000,-81334.00000000,-339603.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000
25%,50000.00000000,1.00000000,1.00000000,1.00000000,28.00000000,-1.00000000,-1.00000000,-1.00000000,-1.00000000,-1.00000000,-1.00000000,3509.75000000,2989.75000000,2599.75000000,2376.75000000,1805.00000000,1255.75000000,1000.00000000,832.00000000,390.00000000,300.00000000,241.00000000,107.75000000,0.00000000
50%,140000.00000000,2.00000000,2.00000000,2.00000000,34.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,22355.00000000,21052.50000000,20077.50000000,18969.50000000,18146.00000000,16972.00000000,2100.00000000,2007.00000000,1806.50000000,1500.00000000,1500.00000000,1500.00000000,0.00000000
75%,240000.00000000,2.00000000,2.00000000,2.00000000,41.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,67622.75000000,64337.25000000,60039.25000000,54889.00000000,50340.75000000,49326.25000000,5012.00000000,5000.00000000,4518.50000000,4076.25000000,4060.00000000,4006.00000000,0.00000000
max,1000000.00000000,2.00000000,6.00000000,3.00000000,79.00000000,8.00000000,8.00000000,8.00000000,8.00000000,8.00000000,8.00000000,964511.00000000,983931.00000000,1664089.00000000,891586.00000000,927171.00000000,961664.00000000,873552.00000000,1684259.00000000,896040.00000000,621000.00000000,426529.00000000,528666.00000000,1.00000000


## EDA

### EDA Target

In [ ]:
df_plot = df_train[target].value_counts().reset_index()
df_plot.columns = ['etiqueta', 'cantidad']
df_plot.iplot(
    kind="pie", 
    labels="etiqueta", 
    values="cantidad", 
    textinfo="percent+label", 
    hole=.4, 
    title="Distribución de la Variable Objetivo (Default)",
    theme="solar"
)

### EDA Continuas

In [31]:
import plotly.express as px

# 1. Definimos las variables (Numéricas continuas para los ejes)
ls_features = ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'PAY_AMT1']

# 2. Creamos la matriz de dispersión
# Usamos el template 'plotly_dark' para que sea idéntico al tema 'solar'
fig = px.scatter_matrix(
    df_train,
    dimensions=ls_features,
    color="default payment next month",
    title="Análisis de Correlación y Separabilidad - Práctica 3",
    template="plotly_dark",
    labels={col: col.replace('.', ' ') for col in ls_features}, # Limpia los nombres de ejes
    color_discrete_sequence=['#2ecc71', '#e74c3c'] # Verde para No Default, Rojo para Default
)

# 3. Ajustes de estilo para que se vea como una matriz profesional
fig.update_traces(
    diagonal_visible=True, # Muestra los histogramas en la diagonal
    showupperhalf=False,   # Opcional: oculta la mitad duplicada para mayor claridad
    marker=dict(size=3, opacity=0.5) # Puntos pequeños para evitar amontonamiento
)

# 4. Forzamos el tamaño para que no se vea "aplastada"
fig.update_layout(
    width=900, 
    height=900,
    font=dict(size=10)
)

fig.show()

In [ ]:
for col in ls_cont:
    fig = px.histogram(
        df_train, 
        x=col, 
        title=f"Distribución de {col}",
        nbins=30,
        template="plotly_dark",
        color_discrete_sequence=['#ff9933'] 
    )
    fig.show()

### EDA Discretas

In [ ]:
for col in ls_disc:
    df_temp = df_train[col].value_counts().reset_index()
    df_temp.columns = ['labels_col', 'values_col']

    df_temp.iplot(
        kind='pie', 
        labels='labels_col', 
        values='values_col',  
        title=f'Distribución por {col}',
        theme='solar'
    )

### EDA Ordinales

In [19]:
for col in ls_ord:
    # Forzamos a que sean 1D usando values.flatten()
    # Esto elimina cualquier dimensión extra (21000, 1) -> (21000,)
    indice = df_train[col].values.flatten()
    columna_target = df_train[target].values.flatten()
    
    # Creamos la tabla de contingencia
    df_cross = pd.crosstab(indice, columna_target).reset_index()
    
    # Renombramos las columnas para que el melt no se confunda
    # 'index' es el valor de la variable de pago, y luego las clases 0 y 1
    df_cross.columns = [col, '0', '1']
    
    # Convertimos a formato largo para la gráfica
    df_melt = df_cross.melt(id_vars=col, var_name='Resultado', value_name='Cantidad')
    
    fig = px.bar(
        df_melt, 
        x=col, 
        y='Cantidad', 
        color='Resultado', 
        barmode='group',
        title=f'Análisis de Impago: {col}',
        template='plotly_dark',
        color_discrete_map={'0': '#2ecc71', '1': '#e74c3c'}
    )
    fig.show()

## Data Cleaning

In [28]:
# 2. Limpieza de Educación (Agrupar 4, 5, 6 y también el 0 que a veces aparece)
# Queremos que 4, 5, 6 y 0 se conviertan todos en 4 ("Otros")
df['EDUCATION'] = df['EDUCATION'].replace([0, 5, 6], 4)

# 3. Definir los grupos de columnas
cols_categoricas = ['SEX', 'MARRIAGE'] # Usaremos OneHotEncoder
cols_ordinales = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6'] # Se quedan como están
cols_numericas = ['LIMIT_BAL', 'AGE', 
                  'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
                  'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

# Nota: EDUCATION ya es numérica (1, 2, 3, 4), así que puede ir en ordinales 
# o numéricas para que el modelo respete su orden.
cols_ordinales.append('EDUCATION')

# 4. Crear el preprocesador
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), cols_numericas),
        ('cat', OneHotEncoder(drop='first'), cols_categoricas),
        ('ord', 'passthrough', cols_ordinales) # "passthrough" significa dejarlas como están
    ])

NameError: name 'df' is not defined